## Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
import joblib
import os

In [3]:
# Path to local CSV
data_path = os.path.join("..", "data", "synthetic_customer_churn_100k.csv")
# Load dataset
df = pd.read_csv(data_path)

In [4]:
# Cell 3: Check data
print(df.shape)
print(df.head())
print(df.dtypes)

(100000, 9)
   CustomerID  Age  Gender  Tenure  MonthlyCharges        Contract  \
0           1   56  Female      68          147.58        Two year   
1           2   69    Male      32           22.54  Month-to-month   
2           3   46  Female      10           52.47        One year   
3           4   32    Male      22          109.67  Month-to-month   
4           5   60  Female      54          130.98  Month-to-month   

      PaymentMethod  TotalCharges Churn  
0     Bank transfer      10052.03    No  
1      Mailed check        686.78    No  
2  Electronic check        537.88    No  
3      Mailed check       2390.04   Yes  
4       Credit card       7081.28    No  
CustomerID          int64
Age                 int64
Gender                str
Tenure              int64
MonthlyCharges    float64
Contract              str
PaymentMethod         str
TotalCharges      float64
Churn                 str
dtype: object


In [5]:
# Cell 4: Drop CustomerID
df = df.drop('CustomerID', axis=1)

In [10]:
# Cell 5: Define columns
target = 'Churn'

categorical_cols = df.select_dtypes(include=['object', 'string']).columns.tolist()
categorical_cols.remove(target)

numerical_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()

print(f"Categorical: {categorical_cols}")
print(f"Numerical: {numerical_cols}")

Categorical: ['Gender', 'Contract', 'PaymentMethod']
Numerical: ['Age', 'Tenure', 'MonthlyCharges', 'TotalCharges']


In [11]:
# Cell 6: Encode target
df['Churn'] = df['Churn'].map({'No': 0, 'Yes': 1})

In [12]:
# Cell 7: Label encode categoricals for XGBoost/LightGBM
df_encoded = df.copy()
le_dict = {}

for col in categorical_cols:
    le = LabelEncoder()
    df_encoded[col] = le.fit_transform(df_encoded[col])
    le_dict[col] = le

In [13]:
# Cell 8: Prepare X and y
X_encoded = df_encoded.drop(target, axis=1)
y = df_encoded[target]

X_raw = df.drop(target, axis=1)

In [14]:
# Cell 9: Train-test split
X_enc_train, X_enc_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42, stratify=y
)

X_raw_train, X_raw_test, _, _ = train_test_split(
    X_raw, y, test_size=0.2, random_state=42, stratify=y
)

In [15]:
print(f"Train: {X_enc_train.shape}, Test: {X_enc_test.shape}")
print(f"Churn rate in train: {y_train.mean():.4f}")

Train: (80000, 7), Test: (20000, 7)
Churn rate in train: 0.3314


In [16]:
# Cell 10: Save data
os.makedirs('../data/processed', exist_ok=True)
os.makedirs('../models', exist_ok=True)

joblib.dump(X_enc_train, '../data/processed/X_train_encoded.pkl')
joblib.dump(X_enc_test, '../data/processed/X_test_encoded.pkl')
joblib.dump(X_raw_train, '../data/processed/X_train_raw.pkl')
joblib.dump(X_raw_test, '../data/processed/X_test_raw.pkl')
joblib.dump(y_train, '../data/processed/y_train.pkl')
joblib.dump(y_test, '../data/processed/y_test.pkl')
joblib.dump(categorical_cols, '../data/processed/cat_features.pkl')
joblib.dump(le_dict, '../models/label_encoders.pkl')

print("Preprocessing complete")

Preprocessing complete
